In [1]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata 
import seaborn as sns
from scipy.stats import zscore
import matplotlib.pyplot as plt
import collections
from natsort import natsorted

from scipy import stats
from scipy import sparse
from sklearn.decomposition import PCA
from umap import UMAP
from statsmodels.stats.multitest import multipletests

from matplotlib.colors import LinearSegmentedColormap

from scroutines.config_plots import *
from scroutines import powerplots # .config_plots import *
from scroutines import pnmf
from scroutines import basicu
from scroutines.gene_modules import GeneModules  


In [2]:
outfigdir = "/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/figures/250409"
!mkdir -p $outfigdir

In [3]:
fin  = "/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/superdupermegaRNA_hasraw.h5ad"
fout = "/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/superdupermegaRNA_hasraw_cheng22_astro.h5ad"

# adata = anndata.read(fin, backed='r')
adata = anndata.read(fin) 

# fix Study: NA should be 2022 RNA
adata.obs['Study'] = adata.obs['Study'].fillna('2022 RNA') #$ dropna()
# use raw counts
adata.X = adata.raw.X
adata

AnnData object with n_obs × n_vars = 396318 × 16572
    obs: 'Age', 'Doublet', 'Doublet Score', 'n_counts', 'n_genes', 'percent_mito', 'sample', 'Type', 'Subclass', 'Class', 'Sample', 'total_counts', 'pct_counts_mt', 'n_genes_by_counts', 'total_counts_mt', 'Doublet?', 'Study', 'Type_leiden'
    var: 'feature_types'

In [4]:
meta = adata.obs.copy()
meta = meta[meta['Study']=='2022 RNA']
meta = meta[meta['Subclass'].isin(['Astro'])]
# meta = meta[meta['Age'].isin(['P28_nr', 'P28_dr'])]
meta

,Age,Doublet,Doublet Score,n_counts,n_genes,percent_mito,sample,Type,Subclass,Class,Sample,total_counts,pct_counts_mt,n_genes_by_counts,total_counts_mt,Doublet?,Study,Type_leiden
AACCAACGTGCAGGAT-1-P8_1a-2022 RNA-1-0,P8,False,0.015945,7199.0,2704.0,0.000833,P8_1a,Astro_A,Astro,Non-neurons,P8_1a,7199.0,0.000833,NaN,NaN,NaN,2022 RNA,Astro_A
AACCACACAGACACCC-1-P8_1a-2022 RNA-1-0,P8,False,0.008629,14131.0,4076.0,0.000142,P8_1a,Astro_A,Astro,Non-neurons,P8_1a,14131.0,0.000142,NaN,NaN,NaN,2022 RNA,Astro_A
AAGACAAAGCAAACAT-1-P8_1a-2022 RNA-1-0,P8,False,0.012605,4837.0,2095.0,0.001034,P8_1a,Astro_A,Astro,Non-neurons,P8_1a,4837.0,0.001034,NaN,NaN,NaN,2022 RNA,Astro_A
AAGTGAACAGGCATGA-1-P8_1a-2022 RNA-1-0,P8,False,0.016672,6017.0,2496.0,0.000499,P8_1a,Astro_A,Astro,Non-neurons,P8_1a,6017.0,0.000499,NaN,NaN,NaN,2022 RNA,Astro_A
ACACAGTAGGGAGGCA-1-P8_1a-2022 RNA-1-0,P8,False,0.008117,7615.0,2792.0,0.000131,P8_1a,Astro_A,Astro,Non-neurons,P8_1a,7615.0,0.000131,NaN,NaN,NaN,2022 RNA,Astro_A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ATCGATGCATGCCATA-1-P38_dr_2b-5,P38_dr,False,0.117371,3916.0,2040.0,0.000766,P38_dr_2b,Astro_A,Astro,Non-neuron,P38_dr_2b,NaN,NaN,NaN,NaN,NaN,2022 RNA,NaN
TCTACCGCATCGGCCA-1-P38_dr_2b-5,P38_dr,False,0.011913,4748.0,2315.0,0.000632,P38_dr_2b,Astro_A,Astro,Non-neuron,P38_dr_2b,NaN,NaN,NaN,NaN,NaN,2022 RNA,NaN
GGGAAGTTCCATGATG-1-P38_dr_2b-5,P38_dr,False,0.117371,4978.0,2396.0,0.003213,P38_dr_2b,Astro_A,Astro,Non-neuron,P38_dr_2b,NaN,NaN,NaN,NaN,NaN,2022 RNA,NaN
AAGCGTTGTGTTGACT-1-P38_dr_2b-5,P38_dr,False,0.013906,2610.0,1562.0,0.001148,P38_dr_2b,Astro_A,Astro,Non-neuron,P38_dr_2b,NaN,NaN,NaN,NaN,NaN,2022 RNA,NaN


In [5]:
meta.groupby(['Subclass', 'Age']).size().unstack()

Age,P6,P8,P10,P12,P12DR,P14,P14DR,P17,P17DR,P21,P21DR,P28,P28_dl,P28_dr,P38,P38_dr
Subclass,,,,,,,,,,,,,,,,
Astro,0,1411,0,0,0,2808,0,2821,0,2516,0,2168,2077,1960,1485,2084
Endo,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Frem1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
L2/3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
L2/3/4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
L4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
L5IT,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
L5NP,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
L5PT,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [6]:
adata = adata[meta.index]
adata

View of AnnData object with n_obs × n_vars = 19330 × 16572
    obs: 'Age', 'Doublet', 'Doublet Score', 'n_counts', 'n_genes', 'percent_mito', 'sample', 'Type', 'Subclass', 'Class', 'Sample', 'total_counts', 'pct_counts_mt', 'n_genes_by_counts', 'total_counts_mt', 'Doublet?', 'Study', 'Type_leiden'
    var: 'feature_types'

In [7]:
# sample_labels = ["-".join(cell.split(' ')[0].split('-')[2:]).replace('-2023', '') for cell in adata.obs.index]
# time_labels = [s[:-1].replace('DR', '') for s in sample_labels]

# # adata.obs['n_counts'] = adata.obs['total_counts'] # adata.obs['nCount_RNA']
# adata.obs['sample'] = sample_labels
# adata.obs['time']   = time_labels

# uniq_samples = natsorted(np.unique(sample_labels))
# uniq_times = natsorted(np.unique(time_labels))

# nr_samples = [s for s in uniq_samples if "DR" not in s]
# dr_samples = [s for s in uniq_samples if "DR" in s]
# print(uniq_times)
# print(nr_samples)
# print(dr_samples)

# adata.obs['sample'] = sample_labels

In [8]:
# filter genes
cond = np.ravel((adata.X>0).sum(axis=0)) > 10 # expressed in more than 10 cells
adata = adata[:,cond]
genes = adata.var.index.values

# counts
x = adata.X
cov = adata.obs['n_counts'].values

# CP10k
# xn = x/cov.reshape(x.shape[0], -1)*1e4
xn = (sparse.diags(1/cov).dot(x))*1e4

# log2(CP10k+1)
# xln = xn.copy()
# xln.data = np.log2(xln.data+1)

In [9]:
adata.layers[    'norm'] = np.array(xn.todense())

# log_xn = np.log2(1+np.array(xn.todense()))
# adata.layers[ 'lognorm'] = log_xn 
# adata.layers['zlognorm'] = zscore(log_xn, axis=0)

In [10]:
adata.write(fout)